In [1]:
# ============================================================================
# MARKETLAB DAY 4 - BACKTESTING & PERFORMANCE ANALYSIS
# ============================================================================

print("🔥 MARKETLAB DAY 4 - BACKTESTING & PERFORMANCE ANALYSIS")
print("="*80)

# Mount Google Drive
print("\n📁 Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted successfully!")

# Import libraries
print("\n📚 Importing libraries...")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta

print("✅ All libraries imported!")

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Paths
BASE_PATH = Path('/content/drive/MyDrive/MarketLab_BEAST')
DATA_PATH = BASE_PATH / 'stock_data'
RESULTS_DAY1 = BASE_PATH / 'results_day1'
RESULTS_DAY2 = BASE_PATH / 'results_day2'
RESULTS_DAY4 = BASE_PATH / 'results_day4'
RESULTS_DAY4.mkdir(exist_ok=True, parents=True)

print(f"\n📂 Created folder: {RESULTS_DAY4}")

print("\n" + "="*80)
print("✅ SETUP COMPLETE!")
print("✅ Ready to run CELL 2!")
print("="*80)

🔥 MARKETLAB DAY 4 - BACKTESTING & PERFORMANCE ANALYSIS

📁 Mounting Google Drive...
Mounted at /content/drive
✅ Drive mounted successfully!

📚 Importing libraries...
✅ All libraries imported!

📂 Created folder: /content/drive/MyDrive/MarketLab_BEAST/results_day4

✅ SETUP COMPLETE!
✅ Ready to run CELL 2!


In [2]:
# ============================================================================
# MARKETLAB DAY 4 - BACKTESTING ANALYSIS
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime, timedelta

print("🔥 MARKETLAB DAY 4 - BACKTESTING")
print("="*80)
print("📊 Simulating trading 2022-2024")
print("🎯 Comparing model vs buy-and-hold strategies")
print("="*80 + "\n")

# ============================================================================
# CONFIGURATION
# ============================================================================

# Top 5 performing stocks for backtesting
BACKTEST_STOCKS = [
    'SUNPHARMA.NS',   # Best: 0.8931
    'ITC.NS',         # Best: 0.8020
    'TCS.NS',         # Best: 0.7341
    'RELIANCE.NS',    # Best: 0.6853
    'MARUTI.NS'       # Newest addition
]

# Backtesting parameters
INITIAL_CAPITAL = 100000  # ₹1,00,000
BACKTEST_START = '2022-01-01'
BACKTEST_END = '2024-12-31'
TRANSACTION_COST = 0.001  # 0.1% per trade

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_stock_data(ticker):
    """Load actual stock price data"""
    path = DATA_PATH / f"{ticker}.csv"

    with open(path, 'r') as f:
        first_line = f.readline()

    if 'Price","Open' in first_line:
        df = pd.read_csv(path)
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
        df = df.set_index('Date')
        df = df.rename(columns={'Price': 'Close', 'Vol.': 'Volume'})
    else:
        df = pd.read_csv(path, skiprows=[1])
        df = df.iloc[1:]
        df['Date'] = pd.to_datetime(df['Price'])
        df = df.set_index('Date')

    df['Close'] = pd.to_numeric(df['Close'], errors='coerce')
    df = df.dropna().sort_index()

    return df[['Close']]

def get_best_model_r2(ticker):
    """Get best R² from Day 1 and Day 2"""
    clean_ticker = ticker.replace('.NS', '')

    # Day 1
    day1_files = list(RESULTS_DAY1.glob(f"{clean_ticker}*.csv"))
    best_r2_day1 = 0
    if len(day1_files) > 0:
        df1 = pd.read_csv(day1_files[0])
        if 'Unnamed: 0' in df1.columns:
            df1 = df1.rename(columns={'Unnamed: 0': 'model'})
        best_r2_day1 = df1['r2'].max()

    # Day 2
    day2_files = list(RESULTS_DAY2.glob(f"{clean_ticker}*.csv"))
    best_r2_day2 = 0
    if len(day2_files) > 0:
        df2 = pd.read_csv(day2_files[0])
        best_r2_day2 = df2['r2'].max()

    return max(best_r2_day1, best_r2_day2)

def calculate_performance_metrics(returns):
    """Calculate comprehensive performance metrics"""

    # Annual metrics
    total_return = (returns + 1).prod() - 1
    n_years = len(returns) / 252
    annual_return = (1 + total_return) ** (1 / n_years) - 1

    # Volatility
    annual_volatility = returns.std() * np.sqrt(252)

    # Sharpe Ratio (risk-free rate = 6% for India)
    risk_free_rate = 0.06
    excess_returns = annual_return - risk_free_rate
    sharpe_ratio = excess_returns / annual_volatility if annual_volatility > 0 else 0

    # Sortino Ratio (downside deviation)
    downside_returns = returns[returns < 0]
    downside_deviation = downside_returns.std() * np.sqrt(252)
    sortino_ratio = excess_returns / downside_deviation if downside_deviation > 0 else 0

    # Maximum Drawdown
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()

    # Win Rate
    win_rate = (returns > 0).sum() / len(returns) if len(returns) > 0 else 0

    return {
        'total_return': total_return,
        'annual_return': annual_return,
        'annual_volatility': annual_volatility,
        'sharpe_ratio': sharpe_ratio,
        'sortino_ratio': sortino_ratio,
        'max_drawdown': max_drawdown,
        'win_rate': win_rate
    }

# ============================================================================
# BACKTESTING SIMULATION
# ============================================================================

def backtest_stock(ticker):
    """Backtest one stock with simple model-based strategy"""

    print(f"\n{'='*80}")
    print(f"Stock: {ticker}")
    print(f"{'='*80}")

    try:
        # Load data
        print("   📥 Loading price data...")
        df = load_stock_data(ticker)

        # Filter to backtest period
        backtest_data = df.loc[BACKTEST_START:BACKTEST_END].copy()

        if len(backtest_data) < 50:
            print(f"   ⚠️  Insufficient data for backtesting")
            return None

        print(f"      ✅ {len(backtest_data)} trading days")

        # Get model performance
        best_r2 = get_best_model_r2(ticker)
        print(f"      ✅ Best model R²: {best_r2:.4f}")

        # === STRATEGY 1: BUY AND HOLD ===
        buy_hold_returns = backtest_data['Close'].pct_change().dropna()

        # === STRATEGY 2: MODEL-BASED TRADING ===
        # Simple strategy: Use R² as confidence score
        # If R² > 0.7: Follow predictions strongly
        # If R² 0.5-0.7: Follow predictions moderately
        # If R² < 0.5: Reduce exposure

        # Simulate predictions (simple momentum based on model quality)
        backtest_data['returns'] = backtest_data['Close'].pct_change()
        backtest_data['ma_5'] = backtest_data['Close'].rolling(5).mean()
        backtest_data['ma_20'] = backtest_data['Close'].rolling(20).mean()

        # Trading signal based on moving average crossover (proxy for model predictions)
        backtest_data['signal'] = 0
        backtest_data.loc[backtest_data['ma_5'] > backtest_data['ma_20'], 'signal'] = 1  # Buy
        backtest_data.loc[backtest_data['ma_5'] <= backtest_data['ma_20'], 'signal'] = -1  # Sell

        # Scale signal by model confidence (R²)
        model_confidence = min(best_r2, 1.0)
        backtest_data['position'] = backtest_data['signal'] * model_confidence

        # Calculate strategy returns
        backtest_data['strategy_returns'] = backtest_data['position'].shift(1) * backtest_data['returns']

        # Apply transaction costs
        backtest_data['trades'] = backtest_data['position'].diff().abs()
        backtest_data['transaction_costs'] = backtest_data['trades'] * TRANSACTION_COST
        backtest_data['strategy_returns_net'] = backtest_data['strategy_returns'] - backtest_data['transaction_costs']

        # Clean data
        strategy_returns = backtest_data['strategy_returns_net'].dropna()

        # === CALCULATE METRICS ===
        print("\n   📊 Calculating performance metrics...")

        buy_hold_metrics = calculate_performance_metrics(buy_hold_returns)
        strategy_metrics = calculate_performance_metrics(strategy_returns)

        print("\n   📈 BUY & HOLD:")
        print(f"      • Total Return: {buy_hold_metrics['total_return']*100:+.2f}%")
        print(f"      • Annual Return: {buy_hold_metrics['annual_return']*100:+.2f}%")
        print(f"      • Sharpe Ratio: {buy_hold_metrics['sharpe_ratio']:.2f}")
        print(f"      • Max Drawdown: {buy_hold_metrics['max_drawdown']*100:.2f}%")

        print("\n   🤖 MODEL STRATEGY:")
        print(f"      • Total Return: {strategy_metrics['total_return']*100:+.2f}%")
        print(f"      • Annual Return: {strategy_metrics['annual_return']*100:+.2f}%")
        print(f"      • Sharpe Ratio: {strategy_metrics['sharpe_ratio']:.2f}")
        print(f"      • Max Drawdown: {strategy_metrics['max_drawdown']*100:.2f}%")

        # Calculate outperformance
        outperformance = strategy_metrics['annual_return'] - buy_hold_metrics['annual_return']
        print(f"\n   🏆 Outperformance: {outperformance*100:+.2f}% annually")

        # === CREATE VISUALIZATIONS ===
        print("\n   🎨 Creating visualizations...")

        # Equity curves
        buy_hold_equity = (1 + buy_hold_returns).cumprod() * INITIAL_CAPITAL
        strategy_equity = (1 + strategy_returns).cumprod() * INITIAL_CAPITAL

        fig, axes = plt.subplots(3, 1, figsize=(14, 12))

        # Plot 1: Price and signals
        axes[0].plot(backtest_data.index, backtest_data['Close'], label='Price', linewidth=1.5)
        axes[0].plot(backtest_data.index, backtest_data['ma_5'], label='MA(5)', alpha=0.7)
        axes[0].plot(backtest_data.index, backtest_data['ma_20'], label='MA(20)', alpha=0.7)
        buy_signals = backtest_data[backtest_data['signal'] == 1]
        sell_signals = backtest_data[backtest_data['signal'] == -1]
        axes[0].scatter(buy_signals.index, buy_signals['Close'], color='green', marker='^', s=50, label='Buy Signal', alpha=0.6)
        axes[0].scatter(sell_signals.index, sell_signals['Close'], color='red', marker='v', s=50, label='Sell Signal', alpha=0.6)
        axes[0].set_title(f'{ticker.replace(".NS", "")} - Price & Trading Signals', fontweight='bold')
        axes[0].set_ylabel('Price (₹)')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Plot 2: Equity curves
        axes[1].plot(buy_hold_equity.index, buy_hold_equity, label='Buy & Hold', linewidth=2)
        axes[1].plot(strategy_equity.index, strategy_equity, label='Model Strategy', linewidth=2)
        axes[1].axhline(y=INITIAL_CAPITAL, color='black', linestyle='--', alpha=0.5, label='Initial Capital')
        axes[1].set_title('Equity Curves Comparison', fontweight='bold')
        axes[1].set_ylabel('Portfolio Value (₹)')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        # Plot 3: Drawdown
        buy_hold_cumulative = (1 + buy_hold_returns).cumprod()
        buy_hold_drawdown = (buy_hold_cumulative - buy_hold_cumulative.expanding().max()) / buy_hold_cumulative.expanding().max()

        strategy_cumulative = (1 + strategy_returns).cumprod()
        strategy_drawdown = (strategy_cumulative - strategy_cumulative.expanding().max()) / strategy_cumulative.expanding().max()

        axes[2].fill_between(buy_hold_drawdown.index, buy_hold_drawdown * 100, 0, alpha=0.3, label='Buy & Hold')
        axes[2].fill_between(strategy_drawdown.index, strategy_drawdown * 100, 0, alpha=0.3, label='Model Strategy')
        axes[2].set_title('Drawdown Analysis', fontweight='bold')
        axes[2].set_ylabel('Drawdown (%)')
        axes[2].set_xlabel('Date')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        clean_ticker = ticker.replace('.NS', '')
        plt.savefig(RESULTS_DAY4 / f'{clean_ticker}_backtest.png', dpi=300, bbox_inches='tight')
        plt.close()

        print(f"   ✅ Visualizations saved!")

        return {
            'ticker': ticker,
            'best_r2': best_r2,
            'buy_hold': buy_hold_metrics,
            'strategy': strategy_metrics,
            'outperformance': outperformance
        }

    except Exception as e:
        print(f"   ❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

# ============================================================================
# RUN BACKTESTS
# ============================================================================

print("🚀 Starting Backtesting...\n")

backtest_results = {}
for stock in BACKTEST_STOCKS:
    result = backtest_stock(stock)
    if result:
        backtest_results[stock] = result

# ============================================================================
# SUMMARY & COMPARISON
# ============================================================================

if len(backtest_results) > 0:
    print("\n" + "="*80)
    print("📊 BACKTESTING SUMMARY")
    print("="*80)

    # Create summary table
    summary_data = []
    for stock, data in backtest_results.items():
        summary_data.append({
            'Stock': stock.replace('.NS', ''),
            'Model_R²': data['best_r2'],
            'BH_Return_%': data['buy_hold']['annual_return'] * 100,
            'Strategy_Return_%': data['strategy']['annual_return'] * 100,
            'Outperformance_%': data['outperformance'] * 100,
            'BH_Sharpe': data['buy_hold']['sharpe_ratio'],
            'Strategy_Sharpe': data['strategy']['sharpe_ratio'],
            'BH_MaxDD_%': data['buy_hold']['max_drawdown'] * 100,
            'Strategy_MaxDD_%': data['strategy']['max_drawdown'] * 100
        })

    summary_df = pd.DataFrame(summary_data)

    print(f"\n{summary_df.to_string(index=False)}")

    # Save summary
    summary_df.to_csv(RESULTS_DAY4 / 'backtest_summary.csv', index=False)

    # Overall statistics
    print("\n" + "="*80)
    print("📈 OVERALL STATISTICS")
    print("="*80)
    print(f"Average Buy & Hold Return: {summary_df['BH_Return_%'].mean():.2f}% annually")
    print(f"Average Strategy Return: {summary_df['Strategy_Return_%'].mean():.2f}% annually")
    print(f"Average Outperformance: {summary_df['Outperformance_%'].mean():+.2f}% annually")
    print(f"Average Sharpe Improvement: {(summary_df['Strategy_Sharpe'].mean() - summary_df['BH_Sharpe'].mean()):+.2f}")
    print(f"Stocks with positive outperformance: {(summary_df['Outperformance_%'] > 0).sum()}/{len(summary_df)}")

    # Create comparison visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Returns comparison
    x = np.arange(len(summary_df))
    width = 0.35
    axes[0, 0].bar(x - width/2, summary_df['BH_Return_%'], width, label='Buy & Hold', alpha=0.8)
    axes[0, 0].bar(x + width/2, summary_df['Strategy_Return_%'], width, label='Model Strategy', alpha=0.8)
    axes[0, 0].set_ylabel('Annual Return (%)')
    axes[0, 0].set_title('Returns Comparison', fontweight='bold')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(summary_df['Stock'], rotation=45)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3, axis='y')

    # Sharpe comparison
    axes[0, 1].bar(x - width/2, summary_df['BH_Sharpe'], width, label='Buy & Hold', alpha=0.8)
    axes[0, 1].bar(x + width/2, summary_df['Strategy_Sharpe'], width, label='Model Strategy', alpha=0.8)
    axes[0, 1].set_ylabel('Sharpe Ratio')
    axes[0, 1].set_title('Risk-Adjusted Returns', fontweight='bold')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(summary_df['Stock'], rotation=45)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3, axis='y')

    # Outperformance
    colors = ['green' if x > 0 else 'red' for x in summary_df['Outperformance_%']]
    axes[1, 0].barh(summary_df['Stock'], summary_df['Outperformance_%'], color=colors, alpha=0.7)
    axes[1, 0].set_xlabel('Outperformance (%)')
    axes[1, 0].set_title('Strategy vs Buy & Hold', fontweight='bold')
    axes[1, 0].axvline(x=0, color='black', linestyle='--', linewidth=1)
    axes[1, 0].grid(True, alpha=0.3, axis='x')

    # Model R² vs Outperformance
    axes[1, 1].scatter(summary_df['Model_R²'], summary_df['Outperformance_%'], s=100, alpha=0.6)
    for idx, row in summary_df.iterrows():
        axes[1, 1].annotate(row['Stock'], (row['Model_R²'], row['Outperformance_%']),
                           fontsize=8, alpha=0.7)
    axes[1, 1].set_xlabel('Model R²')
    axes[1, 1].set_ylabel('Outperformance (%)')
    axes[1, 1].set_title('Model Quality vs Trading Performance', fontweight='bold')
    axes[1, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(RESULTS_DAY4 / 'backtest_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()

    print(f"\n✅ Comparison charts saved!")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("🎉 DAY 4 COMPLETE!")
print("="*80)
print(f"✅ Stocks backtested: {len(backtest_results)}/5")
print(f"✅ Period: 2022-2024 (3 years)")
print(f"✅ Initial capital: ₹{INITIAL_CAPITAL:,}")
if len(backtest_results) > 0:
    print(f"✅ Average outperformance: {summary_df['Outperformance_%'].mean():+.2f}% annually")
print(f"✅ Files created: {len(list(RESULTS_DAY4.glob('*')))}")
print("="*80)
print(f"\n📂 Results saved in: /MarketLab_BEAST/results_day4/")
print("\n📊 Key files created:")
print("   • backtest_summary.csv")
print("   • backtest_comparison.png")
print("   • Individual stock backtest charts (5 files)")
print("\n🎯 KEY FINDINGS:")
if len(backtest_results) > 0:
    best_stock = summary_df.loc[summary_df['Outperformance_%'].idxmax(), 'Stock']
    best_outperf = summary_df['Outperformance_%'].max()
    print(f"   • Best performer: {best_stock} ({best_outperf:+.2f}% outperformance)")
    print(f"   • Average Sharpe ratio: {summary_df['Strategy_Sharpe'].mean():.2f}")
    print(f"   • Success rate: {(summary_df['Outperformance_%'] > 0).sum()}/{len(summary_df)} stocks")

print("\n✅ READY FOR DAY 5 - GEOPOLITICAL INTELLIGENCE!")

🔥 MARKETLAB DAY 4 - BACKTESTING
📊 Simulating trading 2022-2024
🎯 Comparing model vs buy-and-hold strategies

🚀 Starting Backtesting...


Stock: SUNPHARMA.NS
   📥 Loading price data...
      ✅ 738 trading days
      ✅ Best model R²: 0.9986

   📊 Calculating performance metrics...

   📈 BUY & HOLD:
      • Total Return: +128.90%
      • Annual Return: +32.73%
      • Sharpe Ratio: 1.38
      • Max Drawdown: -15.96%

   🤖 MODEL STRATEGY:
      • Total Return: +35.11%
      • Annual Return: +10.84%
      • Sharpe Ratio: 0.25
      • Max Drawdown: -26.08%

   🏆 Outperformance: -21.89% annually

   🎨 Creating visualizations...
   ✅ Visualizations saved!

Stock: ITC.NS
   📥 Loading price data...
      ✅ 738 trading days
      ✅ Best model R²: 0.8020

   📊 Calculating performance metrics...

   📈 BUY & HOLD:
      • Total Return: +145.39%
      • Annual Return: +35.93%
      • Sharpe Ratio: 1.56
      • Max Drawdown: -16.79%

   🤖 MODEL STRATEGY:
      • Total Return: +18.01%
      • Annual Re